# Circuiti quantistici passo passo

In questo notebook vediamo:

1. **Tabelle degli operatori fondamentali**: stato *prima* e *dopo* l'applicazione delle porte a 1 e 2 qubit
2. **Evoluzione passo passo degli stati** in un circuito: come lo stato cambia prima e dopo ogni operatore
3. **Come calcolare l'operatore risultante** di un circuito (sia con Qiskit, sia "a mano" col prodotto di matrici)
4. **Teleportation**: il circuito e la verifica che lo stato arriva a destinazione
5. **Superdense coding**: 2 bit classici trasmessi con 1 qubit
6. **Gioco CHSH**: il vantaggio quantistico rispetto alla migliore strategia classica

**Convenzione di Qiskit (little-endian):** in una stringa come $|q_1 q_0\rangle$ il qubit 0 è quello **più a destra**. Le matrici e le probabilità restituite da Qiskit seguono questo ordinamento.

In [1]:
import numpy as np
from IPython.display import display, Markdown

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, __version__
from qiskit.quantum_info import Statevector, Operator, DensityMatrix, partial_trace
from qiskit.circuit.library import XGate, YGate, ZGate, HGate, SGate, TGate, CXGate, CZGate, SwapGate

print(f"Versione Qiskit: {__version__}")

C:\ProgramData\anaconda3\Lib\site-packages\qiskit\circuit\library\pauli_evolution.py:19: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.5.2)
  import scipy as sc


Versione Qiskit: 2.5.2


## 1. Tabelle: gli operatori classici del quantum, prima e dopo

Per ogni porta a **1 qubit** ($X$, $Y$, $Z$, $H$, $S$, $T$) mostriamo l'effetto sugli stati $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$.
La colonna "prima" è lo stato di ingresso, la colonna "dopo" è $U|\psi\rangle$.

In [2]:
def latex_ket(sv: Statevector) -> str:
    """Rappresentazione LaTeX (inline) di uno Statevector."""
    return "$" + sv.draw("latex_source") + "$"


porte_1q = {"X": XGate(), "Y": YGate(), "Z": ZGate(), "H": HGate(), "S": SGate(), "T": TGate()}
stati_1q = {
    r"$|0\rangle$": Statevector.from_label("0"),
    r"$|1\rangle$": Statevector.from_label("1"),
    r"$|+\rangle$": Statevector.from_label("+"),
    r"$|-\rangle$": Statevector.from_label("-"),
}

righe = ["| Porta | Stato prima | Stato dopo |", "|:---:|:---:|:---:|"]
for nome_porta, porta in porte_1q.items():
    U = Operator(porta)
    for nome_stato, sv in stati_1q.items():
        righe.append(f"| **{nome_porta}** | {nome_stato} = {latex_ket(sv)} | {latex_ket(sv.evolve(U))} |")

display(Markdown("\n".join(righe)))

| Porta | Stato prima | Stato dopo |
|:---:|:---:|:---:|
| **X** | $|0\rangle$ = $ |0\rangle$ | $ |1\rangle$ |
| **X** | $|1\rangle$ = $ |1\rangle$ | $ |0\rangle$ |
| **X** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ |
| **X** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $- \frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ |
| **Y** | $|0\rangle$ = $ |0\rangle$ | $i |1\rangle$ |
| **Y** | $|1\rangle$ = $ |1\rangle$ | $- i |0\rangle$ |
| **Y** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $- \frac{\sqrt{2} i}{2} |0\rangle+\frac{\sqrt{2} i}{2} |1\rangle$ |
| **Y** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2} i}{2} |0\rangle+\frac{\sqrt{2} i}{2} |1\rangle$ |
| **Z** | $|0\rangle$ = $ |0\rangle$ | $ |0\rangle$ |
| **Z** | $|1\rangle$ = $ |1\rangle$ | $- |1\rangle$ |
| **Z** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ |
| **Z** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ |
| **H** | $|0\rangle$ = $ |0\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ |
| **H** | $|1\rangle$ = $ |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ |
| **H** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $ |0\rangle$ |
| **H** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $ |1\rangle$ |
| **S** | $|0\rangle$ = $ |0\rangle$ | $ |0\rangle$ |
| **S** | $|1\rangle$ = $ |1\rangle$ | $i |1\rangle$ |
| **S** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2} i}{2} |1\rangle$ |
| **S** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2} i}{2} |1\rangle$ |
| **T** | $|0\rangle$ = $ |0\rangle$ | $ |0\rangle$ |
| **T** | $|1\rangle$ = $ |1\rangle$ | $(\frac{\sqrt{2}}{2} + \frac{\sqrt{2} i}{2}) |1\rangle$ |
| **T** | $|+\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+(\frac{1}{2} + \frac{i}{2}) |1\rangle$ |
| **T** | $|-\rangle$ = $\frac{\sqrt{2}}{2} |0\rangle- \frac{\sqrt{2}}{2} |1\rangle$ | $\frac{\sqrt{2}}{2} |0\rangle+(- \frac{1}{2} - \frac{i}{2}) |1\rangle$ |

Stessa tabella per le porte a **2 qubit**: CNOT (controllo $= q_0$, target $= q_1$), CZ e SWAP, applicate ai quattro stati della base computazionale $|q_1 q_0\rangle$.

In [3]:
porte_2q = {"CNOT": CXGate(), "CZ": CZGate(), "SWAP": SwapGate()}
basi_2q = ["00", "01", "10", "11"]  # etichette |q1 q0>

righe = ["| Porta | Stato prima | Stato dopo |", "|:---:|:---:|:---:|"]
for nome_porta, porta in porte_2q.items():
    U = Operator(porta)
    for etichetta in basi_2q:
        sv = Statevector.from_label(etichetta)
        righe.append(f"| **{nome_porta}** | {latex_ket(sv)} | {latex_ket(sv.evolve(U))} |")

display(Markdown("\n".join(righe)))

| Porta | Stato prima | Stato dopo |
|:---:|:---:|:---:|
| **CNOT** | $ |00\rangle$ | $ |00\rangle$ |
| **CNOT** | $ |01\rangle$ | $ |11\rangle$ |
| **CNOT** | $ |10\rangle$ | $ |10\rangle$ |
| **CNOT** | $ |11\rangle$ | $ |01\rangle$ |
| **CZ** | $ |00\rangle$ | $ |00\rangle$ |
| **CZ** | $ |01\rangle$ | $ |01\rangle$ |
| **CZ** | $ |10\rangle$ | $ |10\rangle$ |
| **CZ** | $ |11\rangle$ | $- |11\rangle$ |
| **SWAP** | $ |00\rangle$ | $ |00\rangle$ |
| **SWAP** | $ |01\rangle$ | $ |10\rangle$ |
| **SWAP** | $ |10\rangle$ | $ |01\rangle$ |
| **SWAP** | $ |11\rangle$ | $ |11\rangle$ |

## 2. Evoluzione passo passo degli stati in un circuito

La funzione `evoluzione_passo_passo` prende un circuito e uno stato iniziale, applica **una porta alla volta** e mostra lo stato prima e dopo ogni operatore. Le barriere (`barrier`) vengono usate solo come separatori visivi e vengono saltate.

In [4]:
def evoluzione_passo_passo(circuito: QuantumCircuit, stato_iniziale: Statevector | None = None) -> Statevector:
    """Applica il circuito una porta alla volta, mostrando lo stato prima e dopo ogni operatore."""
    n = circuito.num_qubits
    stato = stato_iniziale if stato_iniziale is not None else Statevector.from_label("0" * n)
    display(Markdown(f"**Stato iniziale:** {latex_ket(stato)}"))

    passo = 0
    for istruzione in circuito.data:
        if istruzione.operation.name == "barrier":
            continue
        passo += 1
        indici = [circuito.find_bit(q).index for q in istruzione.qubits]
        mini_circuito = QuantumCircuit(n)
        mini_circuito.append(istruzione.operation, indici)
        stato_dopo = stato.evolve(mini_circuito)
        display(Markdown(
            f"**Passo {passo}** — `{istruzione.operation.name.upper()}` su qubit {indici}:<br>"
            f"prima: {latex_ket(stato)} $\\;\\longrightarrow\\;$ dopo: {latex_ket(stato_dopo)}"
        ))
        stato = stato_dopo
    return stato

### Esempio 1: circuito a un qubit ($H \to T \to H \to S$)

In [5]:
esempio1 = QuantumCircuit(1)
esempio1.h(0)
esempio1.t(0)
esempio1.h(0)
esempio1.s(0)

print(esempio1.draw())
stato_finale = evoluzione_passo_passo(esempio1)

   ┌───┐┌───┐┌───┐┌───┐
q: ┤ H ├┤ T ├┤ H ├┤ S ├
   └───┘└───┘└───┘└───┘


**Stato iniziale:** $ |0\rangle$

**Passo 1** — `H` su qubit [0]:<br>prima: $ |0\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$

**Passo 2** — `T` su qubit [0]:<br>prima: $\frac{\sqrt{2}}{2} |0\rangle+\frac{\sqrt{2}}{2} |1\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |0\rangle+(\frac{1}{2} + \frac{i}{2}) |1\rangle$

**Passo 3** — `H` su qubit [0]:<br>prima: $\frac{\sqrt{2}}{2} |0\rangle+(\frac{1}{2} + \frac{i}{2}) |1\rangle$ $\;\longrightarrow\;$ dopo: $(0.8535533906 + 0.3535533906 i) |0\rangle+(0.1464466094 - 0.3535533906 i) |1\rangle$

**Passo 4** — `S` su qubit [0]:<br>prima: $(0.8535533906 + 0.3535533906 i) |0\rangle+(0.1464466094 - 0.3535533906 i) |1\rangle$ $\;\longrightarrow\;$ dopo: $(0.8535533906 + 0.3535533906 i) |0\rangle+(0.3535533906 + 0.1464466094 i) |1\rangle$

### Esempio 2: creazione dello stato di Bell $|\Phi^+\rangle$

Con $H$ su $q_0$ e poi CNOT (controllo $q_0$, target $q_1$) si passa da $|00\rangle$ allo stato entangled $\frac{1}{\sqrt2}(|00\rangle + |11\rangle)$.

In [6]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)

print(bell.draw())
stato_bell = evoluzione_passo_passo(bell)

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


**Stato iniziale:** $ |00\rangle$

**Passo 1** — `H` su qubit [0]:<br>prima: $ |00\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |01\rangle$

**Passo 2** — `CX` su qubit [0, 1]:<br>prima: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |01\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |11\rangle$

## 3. Come calcolare l'operatore risultante di un circuito

Un circuito con porte $U_1, U_2, \dots, U_k$ (nell'ordine in cui vengono applicate, da sinistra a destra nel disegno) equivale a un **unico operatore unitario**

$$U_{\text{tot}} = U_k \cdots U_2 \, U_1$$

Attenzione all'ordine: **l'ultima porta applicata è la prima nel prodotto di matrici** (le matrici agiscono sul vettore da destra: $U_{\text{tot}}|\psi\rangle = U_k(\cdots(U_1|\psi\rangle))$).

In Qiskit basta `Operator.from_circuit(circuito)`; qui verifichiamo che coincide col prodotto fatto a mano.

In [7]:
# Operatore risultante calcolato da Qiskit (esempio 1: H -> T -> H -> S)
U_qiskit = Operator.from_circuit(esempio1)
display(Markdown("**Operatore risultante secondo Qiskit** ($U_{tot} = S\\,H\\,T\\,H$):"))
display(U_qiskit.draw("latex"))

# Stesso operatore calcolato a mano: prodotto delle matrici in ORDINE INVERSO di applicazione
H = Operator(HGate()).data
T = Operator(TGate()).data
S = Operator(SGate()).data
U_a_mano = S @ H @ T @ H  # ultima porta applicata (S) piu' a sinistra

print("Il prodotto a mano coincide con Operator.from_circuit?",
      np.allclose(U_qiskit.data, U_a_mano))

# Funziona anche con piu' qubit: operatore del circuito di Bell
display(Markdown("**Operatore risultante del circuito di Bell** ($U = \\mathrm{CNOT}\\,(I \\otimes H)$):"))
display(Operator.from_circuit(bell).draw("latex"))

**Operatore risultante secondo Qiskit** ($U_{tot} = S\,H\,T\,H$):

<IPython.core.display.Latex object>

Il prodotto a mano coincide con Operator.from_circuit? True


**Operatore risultante del circuito di Bell** ($U = \mathrm{CNOT}\,(I \otimes H)$):

<IPython.core.display.Latex object>

## 4. Teleportation

Alice vuole trasferire lo stato $|\psi\rangle$ del qubit $Q$ ($q_0$) a Bob, usando una coppia di Bell condivisa ($A = q_1$ per Alice, $B = q_2$ per Bob) e 2 bit classici.

Il protocollo standard prevede: misura di Bell sui qubit $Q$ e $A$, invio dei due esiti a Bob, correzioni $X$ e/o $Z$ da parte di Bob. Ecco il circuito con le misure e le operazioni condizionate:

In [8]:
qubit = QuantumRegister(1, "Q")   # stato da teleportare (Alice)
ebit_A = QuantumRegister(1, "A")  # meta' della coppia di Bell (Alice)
ebit_B = QuantumRegister(1, "B")  # meta' della coppia di Bell (Bob)
bit_a = ClassicalRegister(1, "a")
bit_b = ClassicalRegister(1, "b")

teleport = QuantumCircuit(qubit, ebit_A, ebit_B, bit_a, bit_b)

# Creazione della coppia di Bell condivisa tra Alice e Bob
teleport.h(ebit_A)
teleport.cx(ebit_A, ebit_B)
teleport.barrier()

# Misura di Bell di Alice su Q e A
teleport.cx(qubit, ebit_A)
teleport.h(qubit)
teleport.measure(ebit_A, bit_a)
teleport.measure(qubit, bit_b)
teleport.barrier()

# Correzioni di Bob condizionate ai bit classici ricevuti
with teleport.if_test((bit_a, 1)):
    teleport.x(ebit_B)
with teleport.if_test((bit_b, 1)):
    teleport.z(ebit_B)

print(teleport.draw())

                ░      ┌───┐┌─┐ ░                                              »
  Q: ───────────░───■──┤ H ├┤M├─░──────────────────────────────────────────────»
     ┌───┐      ░ ┌─┴─┐└┬─┬┘└╥┘ ░                                              »
  A: ┤ H ├──■───░─┤ X ├─┤M├──╫──░──────────────────────────────────────────────»
     └───┘┌─┴─┐ ░ └───┘ └╥┘  ║  ░ ┌────── ┌───┐ ───────┐ ┌────── ┌───┐ ───────┐»
  B: ─────┤ X ├─░────────╫───╫──░─┤ If-0  ┤ X ├  End-0 ├─┤ If-0  ┤ Z ├  End-0 ├»
          └───┘ ░        ║   ║  ░ └──╥─── └───┘ ───────┘ └──╥─── └───┘ ───────┘»
                         ║   ║    ┌──╨──┐                   ║                  »
a: 1/════════════════════╩═══╬════╡ 0x1 ╞═══════════════════╬══════════════════»
                         0   ║    └─────┘                ┌──╨──┐               »
b: 1/════════════════════════╩═══════════════════════════╡ 0x1 ╞═══════════════»
                             0                           └─────┘               »
«      
«  Q: ─
«      
«  A

### Evoluzione passo passo (versione a misura differita)

Per seguire lo stato con `Statevector` sostituiamo misure + correzioni classiche con porte controllate (**principio della misura differita**): $\text{CX}(A, B)$ al posto della correzione $X$ e $\text{CZ}(Q, B)$ al posto della correzione $Z$. Il risultato su $B$ è identico.

Prepariamo su $Q$ uno stato arbitrario $|\psi\rangle = R_z(\varphi) R_y(\vartheta)|0\rangle$ e seguiamo tutta l'evoluzione:

In [9]:
theta, phi = 0.9, 0.4  # parametri dello stato da teleportare

# Stato |psi> atteso su un singolo qubit (per la verifica finale)
prep = QuantumCircuit(1)
prep.ry(theta, 0)
prep.rz(phi, 0)
psi = Statevector.from_instruction(prep)
display(Markdown(f"**Stato da teleportare** $|\\psi\\rangle$ = {latex_ket(psi)}"))

# Circuito di teleportation unitario: q0 = Q, q1 = A, q2 = B
teleport_u = QuantumCircuit(3)
teleport_u.ry(theta, 0)
teleport_u.rz(phi, 0)      # preparazione di |psi> su Q
teleport_u.barrier()
teleport_u.h(1)
teleport_u.cx(1, 2)        # coppia di Bell tra A e B
teleport_u.barrier()
teleport_u.cx(0, 1)
teleport_u.h(0)            # misura di Bell (parte unitaria)
teleport_u.barrier()
teleport_u.cx(1, 2)        # correzione X differita
teleport_u.cz(0, 2)        # correzione Z differita

print(teleport_u.draw())
stato_finale_tp = evoluzione_passo_passo(teleport_u)

**Stato da teleportare** $|\psi\rangle$ = $(0.8824981101 - 0.1788912232 i) |0\rangle+(0.4262951825 + 0.0864143116 i) |1\rangle$

     ┌─────────┐┌─────────┐ ░            ░      ┌───┐ ░         
q_0: ┤ Ry(0.9) ├┤ Rz(0.4) ├─░────────────░───■──┤ H ├─░───────■─
     └─────────┘└─────────┘ ░ ┌───┐      ░ ┌─┴─┐└───┘ ░       │ 
q_1: ───────────────────────░─┤ H ├──■───░─┤ X ├──────░───■───┼─
                            ░ └───┘┌─┴─┐ ░ └───┘      ░ ┌─┴─┐ │ 
q_2: ───────────────────────░──────┤ X ├─░────────────░─┤ X ├─■─
                            ░      └───┘ ░            ░ └───┘   


**Stato iniziale:** $ |000\rangle$

**Passo 1** — `RY` su qubit [0]:<br>prima: $ |000\rangle$ $\;\longrightarrow\;$ dopo: $0.9004471024 |000\rangle+0.4349655341 |001\rangle$

**Passo 2** — `RZ` su qubit [0]:<br>prima: $0.9004471024 |000\rangle+0.4349655341 |001\rangle$ $\;\longrightarrow\;$ dopo: $(0.8824981101 - 0.1788912232 i) |000\rangle+(0.4262951825 + 0.0864143116 i) |001\rangle$

...

**Passo 4** — `CX` su qubit [1, 2]:<br>prima: $(0.6240203981 - 0.126495197 i) |000\rangle+(0.3014362143 + 0.0611041457 i) |001\rangle+(0.6240203981 - 0.126495197 i) |010\rangle+(0.3014362143 + 0.0611041457 i) |011\rangle$ $\;\longrightarrow\;$ dopo: $(0.6240203981 - 0.126495197 i) |000\rangle+(0.3014362143 + 0.0611041457 i) |001\rangle+(0.6240203981 - 0.126495197 i) |110\rangle+(0.3014362143 + 0.0611041457 i) |111\rangle$

**Passo 5** — `CX` su qubit [0, 1]:<br>prima: $(0.6240203981 - 0.126495197 i) |000\rangle+(0.3014362143 + 0.0611041457 i) |001\rangle+(0.6240203981 - 0.126495197 i) |110\rangle+(0.3014362143 + 0.0611041457 i) |111\rangle$ $\;\longrightarrow\;$ dopo: $(0.6240203981 - 0.126495197 i) |000\rangle+(0.3014362143 + 0.0611041457 i) |011\rangle+(0.3014362143 + 0.0611041457 i) |101\rangle+(0.6240203981 - 0.126495197 i) |110\rangle$

**Passo 6** — `H` su qubit [0]:<br>prima: $(0.6240203981 - 0.126495197 i) |000\rangle+(0.3014362143 + 0.0611041457 i) |011\rangle+(0.3014362143 + 0.0611041457 i) |101\rangle+(0.6240203981 - 0.126495197 i) |110\rangle$ $\;\longrightarrow\;$ dopo: $(0.4412490551 - 0.0894456116 i) |000\rangle+(0.4412490551 - 0.0894456116 i) |001\rangle+(0.2131475912 + 0.0432071558 i) |010\rangle+(-0.2131475912 - 0.0432071558 i) |011\rangle+(0.2131475912 + 0.0432071558 i) |100\rangle+(-0.2131475912 - 0.0432071558 i) |101\rangle+(0.4412490551 - 0.0894456116 i) |110\rangle+(0.4412490551 - 0.0894456116 i) |111\rangle$

**Passo 7** — `CX` su qubit [1, 2]:<br>prima: $(0.4412490551 - 0.0894456116 i) |000\rangle+(0.4412490551 - 0.0894456116 i) |001\rangle+(0.2131475912 + 0.0432071558 i) |010\rangle+(-0.2131475912 - 0.0432071558 i) |011\rangle+(0.2131475912 + 0.0432071558 i) |100\rangle+(-0.2131475912 - 0.0432071558 i) |101\rangle+(0.4412490551 - 0.0894456116 i) |110\rangle+(0.4412490551 - 0.0894456116 i) |111\rangle$ $\;\longrightarrow\;$ dopo: $(0.4412490551 - 0.0894456116 i) |000\rangle+(0.4412490551 - 0.0894456116 i) |001\rangle+(0.4412490551 - 0.0894456116 i) |010\rangle+(0.4412490551 - 0.0894456116 i) |011\rangle+(0.2131475912 + 0.0432071558 i) |100\rangle+(-0.2131475912 - 0.0432071558 i) |101\rangle+(0.2131475912 + 0.0432071558 i) |110\rangle+(-0.2131475912 - 0.0432071558 i) |111\rangle$

**Passo 8** — `CZ` su qubit [0, 2]:<br>prima: $(0.4412490551 - 0.0894456116 i) |000\rangle+(0.4412490551 - 0.0894456116 i) |001\rangle+(0.4412490551 - 0.0894456116 i) |010\rangle+(0.4412490551 - 0.0894456116 i) |011\rangle+(0.2131475912 + 0.0432071558 i) |100\rangle+(-0.2131475912 - 0.0432071558 i) |101\rangle+(0.2131475912 + 0.0432071558 i) |110\rangle+(-0.2131475912 - 0.0432071558 i) |111\rangle$ $\;\longrightarrow\;$ dopo: $(0.4412490551 - 0.0894456116 i) |000\rangle+(0.4412490551 - 0.0894456116 i) |001\rangle+(0.4412490551 - 0.0894456116 i) |010\rangle+(0.4412490551 - 0.0894456116 i) |011\rangle+(0.2131475912 + 0.0432071558 i) |100\rangle+(0.2131475912 + 0.0432071558 i) |101\rangle+(0.2131475912 + 0.0432071558 i) |110\rangle+(0.2131475912 + 0.0432071558 i) |111\rangle$

In [10]:
# Verifica: lo stato ridotto del qubit di Bob (q2) deve essere esattamente |psi><psi|
rho_bob = partial_trace(stato_finale_tp, [0, 1])
rho_atteso = DensityMatrix(psi)

print("Lo stato di Bob coincide con |psi>?", np.allclose(rho_bob.data, rho_atteso.data))
display(Markdown("**Matrice densità ridotta di Bob:**"))
display(rho_bob.draw("latex"))
display(Markdown("**Matrice densità attesa** $|\\psi\\rangle\\langle\\psi|$:"))
display(rho_atteso.draw("latex"))

Lo stato di Bob coincide con |psi>? True


**Matrice densità ridotta di Bob:**

<IPython.core.display.Latex object>

**Matrice densità attesa** $|\psi\rangle\langle\psi|$:

<IPython.core.display.Latex object>

## 5. Superdense coding

Il protocollo inverso della teleportation: con **1 qubit** (più una coppia di Bell condivisa) Alice trasmette **2 bit classici** a Bob.

- Alice e Bob condividono $|\Phi^+\rangle$ (Alice ha $q_0$, Bob ha $q_1$).
- Per il messaggio $m = m_1 m_0$, Alice applica al proprio qubit: $X$ se $m_1 = 1$, $Z$ se $m_0 = 1$, e invia il qubit a Bob.
- Bob decodifica con CNOT e $H$, poi misura: ottiene esattamente $m_1 m_0$.

In [11]:
def circuito_superdense(messaggio: str) -> QuantumCircuit:
    """Superdense coding per un messaggio di 2 bit 'm1m0'. q0 = Alice, q1 = Bob."""
    qc = QuantumCircuit(2)
    # Coppia di Bell condivisa
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    # Codifica di Alice sul proprio qubit
    if messaggio[0] == "1":
        qc.x(0)
    if messaggio[1] == "1":
        qc.z(0)
    qc.barrier()
    # Decodifica di Bob (dopo aver ricevuto il qubit di Alice)
    qc.cx(0, 1)
    qc.h(0)
    return qc


print(circuito_superdense("11").draw())

for messaggio in ["00", "01", "10", "11"]:
    stato = Statevector.from_label("00").evolve(circuito_superdense(messaggio))
    probabilita = stato.probabilities_dict()
    display(Markdown(
        f"**Messaggio inviato:** `{messaggio}` — stato prima della misura: {latex_ket(stato)} "
        f"— esiti di misura: `{ {k: round(v, 6) for k, v in probabilita.items()} }`"
    ))

     ┌───┐      ░ ┌───┐┌───┐ ░      ┌───┐
q_0: ┤ H ├──■───░─┤ X ├┤ Z ├─░───■──┤ H ├
     └───┘┌─┴─┐ ░ └───┘└───┘ ░ ┌─┴─┐└───┘
q_1: ─────┤ X ├─░────────────░─┤ X ├─────
          └───┘ ░            ░ └───┘     


**Messaggio inviato:** `00` — stato prima della misura: $ |00\rangle$ — esiti di misura: `{np.str_('00'): np.float64(1.0), np.str_('01'): np.float64(0.0)}`

**Messaggio inviato:** `01` — stato prima della misura: $ |01\rangle$ — esiti di misura: `{np.str_('00'): np.float64(0.0), np.str_('01'): np.float64(1.0)}`

**Messaggio inviato:** `10` — stato prima della misura: $ |10\rangle$ — esiti di misura: `{np.str_('10'): np.float64(1.0), np.str_('11'): np.float64(0.0)}`

**Messaggio inviato:** `11` — stato prima della misura: $ |11\rangle$ — esiti di misura: `{np.str_('10'): np.float64(0.0), np.str_('11'): np.float64(1.0)}`

In [12]:
# Evoluzione passo passo per il messaggio "10"
display(Markdown("### Evoluzione passo passo del superdense coding, messaggio `10`"))
_ = evoluzione_passo_passo(circuito_superdense("10"))

### Evoluzione passo passo del superdense coding, messaggio `10`

**Stato iniziale:** $ |00\rangle$

**Passo 1** — `H` su qubit [0]:<br>prima: $ |00\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |01\rangle$

**Passo 2** — `CX` su qubit [0, 1]:<br>prima: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |01\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |11\rangle$

**Passo 3** — `X` su qubit [0]:<br>prima: $\frac{\sqrt{2}}{2} |00\rangle+\frac{\sqrt{2}}{2} |11\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |01\rangle+\frac{\sqrt{2}}{2} |10\rangle$

**Passo 4** — `CX` su qubit [0, 1]:<br>prima: $\frac{\sqrt{2}}{2} |01\rangle+\frac{\sqrt{2}}{2} |10\rangle$ $\;\longrightarrow\;$ dopo: $\frac{\sqrt{2}}{2} |10\rangle+\frac{\sqrt{2}}{2} |11\rangle$

**Passo 5** — `H` su qubit [0]:<br>prima: $\frac{\sqrt{2}}{2} |10\rangle+\frac{\sqrt{2}}{2} |11\rangle$ $\;\longrightarrow\;$ dopo: $ |10\rangle$

## 6. Il gioco CHSH e il vantaggio quantistico

**Regole del gioco:** un arbitro invia bit casuali $x$ ad Alice e $y$ a Bob (che non possono comunicare). Loro rispondono con bit $a$ e $b$. Vincono se

$$a \oplus b = x \wedge y$$

- **Strategia classica ottima:** rispondere sempre $a = b = 0$ vince nei 3 casi su 4 in cui $x \wedge y = 0$: probabilità di vittoria $= 3/4 = 0{,}75$. Nessuna strategia classica (anche con casualità condivisa) può fare meglio.
- **Strategia quantistica:** Alice e Bob condividono lo stato di Bell $|\Phi^+\rangle$ e misurano in basi ruotate:
  - Alice: angolo $\alpha_0 = 0$ se $x=0$, $\alpha_1 = \pi/2$ se $x=1$
  - Bob: angolo $\beta_0 = \pi/4$ se $y=0$, $\beta_1 = -\pi/4$ se $y=1$

  (misurare "nella base ruotata di $\vartheta$" = applicare $R_y(-\vartheta)$ e poi misurare in base computazionale). La probabilità di vittoria diventa $\cos^2(\pi/8) \approx 0{,}853$ per **ogni** coppia $(x, y)$.

In [13]:
angolo_alice = {0: 0.0, 1: np.pi / 2}
angolo_bob = {0: np.pi / 4, 1: -np.pi / 4}


def circuito_chsh(x: int, y: int) -> QuantumCircuit:
    """Stato di Bell + rotazioni di misura: q0 = Alice (bit a), q1 = Bob (bit b)."""
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    qc.ry(-angolo_alice[x], 0)
    qc.ry(-angolo_bob[y], 1)
    return qc


print(circuito_chsh(1, 1).draw())

righe = ["| $x$ | $y$ | $x \\wedge y$ | P(vittoria) quantistica |", "|:---:|:---:|:---:|:---:|"]
p_media = 0.0
for x in (0, 1):
    for y in (0, 1):
        stato = Statevector.from_label("00").evolve(circuito_chsh(x, y))
        # chiave = 'q1 q0' = 'b a'; si vince se a XOR b == x AND y
        p_vittoria = sum(
            p for chiave, p in stato.probabilities_dict().items()
            if (int(chiave[0]) ^ int(chiave[1])) == (x & y)
        )
        p_media += p_vittoria / 4
        righe.append(f"| {x} | {y} | {x & y} | {p_vittoria:.4f} |")

display(Markdown("\n".join(righe)))
display(Markdown(
    f"**Probabilità media di vittoria quantistica: {p_media:.4f}** "
    f"(teorica: $\\cos^2(\\pi/8) = {np.cos(np.pi/8)**2:.4f}$) "
    f"contro il limite classico di **0.75**."
))

     ┌───┐      ░ ┌──────────┐
q_0: ┤ H ├──■───░─┤ Ry(-π/2) ├
     └───┘┌─┴─┐ ░ ├─────────┬┘
q_1: ─────┤ X ├─░─┤ Ry(π/4) ├─
          └───┘ ░ └─────────┘ 


| $x$ | $y$ | $x \wedge y$ | P(vittoria) quantistica |
|:---:|:---:|:---:|:---:|
| 0 | 0 | 0 | 0.8536 |
| 0 | 1 | 0 | 0.8536 |
| 1 | 0 | 0 | 0.8536 |
| 1 | 1 | 1 | 0.8536 |

**Probabilità media di vittoria quantistica: 0.8536** (teorica: $\cos^2(\pi/8) = 0.8536$) contro il limite classico di **0.75**.

In [14]:
# Valore CHSH: S = <A0 B0> + <A0 B1> + <A1 B0> - <A1 B1>
# Classicamente |S| <= 2 (disuguaglianza di Bell-CHSH); il quantum raggiunge 2*sqrt(2) (limite di Tsirelson)
def correlazione(x: int, y: int) -> float:
    """E(a=b) - E(a!=b) per la coppia di basi (x, y)."""
    stato = Statevector.from_label("00").evolve(circuito_chsh(x, y))
    return sum(
        (1 if int(chiave[0]) == int(chiave[1]) else -1) * p
        for chiave, p in stato.probabilities_dict().items()
    )


S = correlazione(0, 0) + correlazione(0, 1) + correlazione(1, 0) - correlazione(1, 1)
print(f"S quantistico  = {S:.6f}")
print(f"2*sqrt(2)      = {2 * np.sqrt(2):.6f}  (limite di Tsirelson, raggiunto)")
print("Limite classico = 2  ->  la disuguaglianza CHSH è violata: vantaggio quantistico!")

# Simulazione con misure ripetute
n_partite = 4000
vittorie = 0
rng = np.random.default_rng(42)
for _ in range(n_partite):
    x, y = rng.integers(0, 2), rng.integers(0, 2)
    stato = Statevector.from_label("00").evolve(circuito_chsh(x, y))
    esito, _ = stato.measure()
    a, b = int(esito[1]), int(esito[0])
    vittorie += int((a ^ b) == (x & y))

print(f"\nSimulazione di {n_partite} partite: frazione di vittorie = {vittorie / n_partite:.4f} "
      f"(atteso ~0.8536, classico <= 0.75)")

S quantistico  = 2.828427
2*sqrt(2)      = 2.828427  (limite di Tsirelson, raggiunto)
Limite classico = 2  ->  la disuguaglianza CHSH è violata: vantaggio quantistico!

Simulazione di 4000 partite: frazione di vittorie = 0.8608 (atteso ~0.8536, classico <= 0.75)


## Riepilogo

- Le **tabelle prima/dopo** mostrano l'azione delle porte fondamentali sugli stati della base (e su $|\pm\rangle$).
- Con `evoluzione_passo_passo` si segue lo stato **prima e dopo ogni operatore** di un circuito.
- L'**operatore risultante** di un circuito è il prodotto delle matrici delle porte **in ordine inverso di applicazione**; in Qiskit: `Operator.from_circuit(circuito)`.
- La **teleportation** trasferisce uno stato quantistico con una coppia di Bell + 2 bit classici (verificato con la traccia parziale).
- Il **superdense coding** trasmette 2 bit classici con 1 qubit + una coppia di Bell.
- Il gioco **CHSH** mostra il vantaggio quantistico: probabilità di vittoria $\cos^2(\pi/8) \approx 0{,}853 > 0{,}75$ e valore $S = 2\sqrt{2} > 2$ (violazione della disuguaglianza di Bell-CHSH).